# OpenAI - Tool Calling

source: [OpenAI tools guide](https://developers.openai.com/api/docs/guides/tools)
Tool calling means:
- use built-in tools
- function calling
- programmatic tool calling
- [tool search](https://developers.openai.com/api/docs/guides/tools-tool-search)
- remote mcp servers

Gotcha (hard-learned):
- function calling (aka tool calling)
    - when you use function calling it's important to adjust the prompt to say to the model that:
        - `You MUST answer ONLY USING the results of the function call.`
        - `If the function call results are empty or insufficient say it explicitly and don't answer using your own knowledge.`
    - without these two points the model can:
        - hallucinate predicting the answer instead of using the function call response
        - provide a ungrounded/partial-grounded answer if the function call response has insufficient details to answer  
- websearch
    - search_context_size: low can be a problem because it returns a low amount of tokens from webpages  
    - gpt-5-nano: small models can use a lot of tokens when using web_search because they're not so good at agentic loop and if they can't find results they keep trying in the wrong way, so a bigger model may be better and use less tokens (but they cost much more)
    - By default summary is None and the reasoning steps content is encrypted, if you want to understand what is happening in the reasoning steps, you can set summary to "concise" or "detailed, using `reasoning={"summary":"concise"}`, this will return a not encrypted summary for each reasoning step   

## Function Calling
source: [Function Calling](https://developers.openai.com/api/docs/guides/function-calling?api-mode=responses)

With the function calling feature you can a JSON Schema of you functions and it can decide to call them instead of generating a message.

If you have many functions you can pair function callign with tool_search to defer the loading of rarely used tools.

This is the standard Function Calling flow:
1. Make a request to the model with __tools__ it could call
2. Receive a __tool call__ from the model (containing the function to call and the input parameters)
3. __Execute code__ on the application side with input from the tool call
4. Make a second request to the model with the __tool output__ (remember to add the reference to the tool call id)
5. Receive a __final response__ from the model (or more tool calls)


In [15]:
from openai import OpenAI
import json

client = OpenAI()

def get_horoscope(sign):
    return f"{sign}: Next Tuesday you will befriend a baby otter."

# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_horoscope",
        "description": "Get today's horoscope for an astrological sign.",
        "parameters": {
            "type": "object",
            "properties": {
                "sign": {
                    "type": "string",
                    "description": "An astrological sign like Taurus or Pisces",
                },
            },
            "required": ["sign"],
        },
    },
]

# Create a running input list we will add to over time to give the model context of what has happened in the conversation so far.
input_list = [{"role": "user", "content": "What is my horoscope? I am a Pisces."}]

# 2. Prompt the model with tools defined
response = client.responses.create(
    model="gpt-5-nano",
    tools=tools,
    input=input_list,
)

# Save function call outputs for subsequent requests
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        print(f"Function call detected: {item.name} with arguments {item.arguments}")
        if item.name == "get_horoscope":
            # 3. Execute the function logic for get_horoscope
            sign = json.loads(item.arguments)["sign"]
            horoscope = get_horoscope(sign)

            # 4. Provide function call results to the model
            input_list.append(
                {
                    "type": "function_call_output",  # this is a special type that tells the model this is the output of a function call
                    "call_id": item.call_id,         # you must provide the call_id to associate the output with the function call
                    "output": horoscope,             # this can be any string or JSON object, but must be serializable to JSON
                }
            )

print("Final input:")
print(*input_list, sep="\n")

response = client.responses.create(
    model="gpt-5-nano",
    instructions="Respond only with a horoscope generated by a tool.", # This is important to prevent the model from hallucinating a horoscope instead of using the tool.
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print(f"Final output: \n{response.model_dump_json(indent=2)}")
print(f"\nOutput text: {response.output_text}")

Function call detected: get_horoscope with arguments {"sign":"Pisces"}
Final input:
{'role': 'user', 'content': 'What is my horoscope? I am a Pisces.'}
ResponseReasoningItem(id='rs_03593596e920e252006a99a0a22a1087d2948873f0be872576', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqmaCjBK_hFWPUTavmggvHCD3Ebo4EUnY2wzZaBC8sS9kYJ0kNumTIpB0t2RhaxZ9cqDRwYfoVREPMgPobiyKeOFzA-tRchHIjGQyb18kFfPQD-r5E7JnK5-lZ9_nzpxFS5VahqwiSAz0sP_QGJZlu9P9egRA8EDiDN9F9vsEozeCGTwyBf_qEYbFNS9WjYggvy1zD8Sp5em9cjWSmf8WSYBYq_V3THSAGNvtR7NGRfyM7BXQIjb-O57MhLLC1HbtLCcWU32r07-zHIsyRFfUCDmzQc4SOgCP0qZxV514NKl-_3uYhtZtFLeu7zT6KfYXxe_mYn2rDxTGLuMJsm5duf7MYUBoObxuy-7AyCzUVrNP6G2yUilOtw9h2AuKlCe6F4eBiOlcyjw60cmPezAapKvUytmAZnqpUDopypsbytcwAtkW0PwIWjvVnJj5UqLQ2lVvLNpyCjm3O0II9yDqkxV_MQmwu221mmUcBTjaWhRISUVAmLWAepkklA2rSxofWN1YXAcSy88XhliylJ-DUsNVBGVF5EcZHWa9G-j4z-ZMwbPQEAlRaBgrqQv131qfscKtPx0JsLtA598MkK5prRHAXnMqpXGn9lXlWF6DSZATq1PBjWjY3Zfh0cM0klSO4i44U0x6xosHQVrPJbJanoQiynbFef2Sznra-tsWy10YE15p0lLwxyWWFL

#### Summary

1. First of all you need to tell the LLM about the tool it can use by passing the tool argument  
```python
    [
        {                                                                               # json schema describing a function that the LLM can ask you to call
            "type": "function",
            "name": "get_horoscope",                                                    # name that LLM will use to identify the function
            "description": "Get today's horoscope for an astrological sign.",           # description used by the LLM to understand if it needs to call this function
            "parameters": {                                                             # parameters accepted by the function
                "type": "object",
                "properties": {
                    "sign": {
                        "type": "string",
                        "description": "An astrological sign like Taurus or Pisces",
                    },
                },
                "required": ["sign"],                                                   # remember to mark the required parameters!
            },
        },
        ...
    ]
```
2. When the model wants to call a function it returns a response containing the following response:
```python
    {
        type='function_call',
        name='get_horoscope',                   # function to call
        arguments='{"sign":"Pisces"}',          # arguments to pass to the function (json)
        call_id='call_QPWUFINJwcKlhwZEN6f2Cn4r' # id that you will insert in the function call output message
    }
```
3. Then we execute the requested function and add the function output to the input passed to the LLM:
```python
    [   ...previous messages, # here you must add all the chat history (both user and LLM messages)
        {
            "type": "function_call_output",
            "call_id": call_id,              # you must provide the call_id of the LLM response containing the function call request 
            "output": function_response,     # this is the function output and can be any string or JSON object, but must be serializable to JSON
        }
    ]
```
4. Once you call the LLM it will read the function_response and provide you an answer or another function call


#### Best practices


0. see [best practices](https://developers.openai.com/api/docs/guides/function-calling?api-mode=responses&strict-mode=enabled#best-practices-for-defining-functions) for more
1. If you have multiple functions that can be called by the LLM it could be useful to implement a facade function like this in your python code to switch the call to the right function:
```python
    def call_function(name, args):
        if name == "get_weather":
            return get_weather(**args)
        if name == "send_email":
            return send_email(**args)
        raise ValueError(f"Unknown function: {name}")
```
2. It's important that you tell the LLM to use ONLY the function_response to create the answer and to explicitly tell the user if the data in the function_response are insufficient, never make up or give partial-grounded answers  
3. You should expose max 20 functions to the LLM, if you need to expose more than 20 functions use `tool_search`
4. Use the tool choice parameter if you need to enforce the usage of a tool
```python
    "tool_choice": {
        "type": "allowed_tools", # auto: (Default) Call zero, one, or multiple functions
                                 # required: Call one or more functions
                                 # forced_function: Call exactly one specific function - tool_choice: {"type": "function", "name": "get_weather"}
                                 # allowed_tools: Restrict the tool calls the model can make to a subset of the tools available to the model
        "mode": "auto",
        "tools": [
            { "type": "function", "name": "..." },
            { "type": "function", "name": "..." }
        ]
    }
```

## Websearch

### GPT-5-nano
gpt-5-nano took 32k tokens to produce a response

In [5]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

response = client.responses.create(
    model="gpt-5-nano",
    tools=[
        {
          "type": "web_search",
          "search_context_size": "low", # how much context from web search results before generate a response
          "filters": {
              "allowed_domains": [ # up to 100
                "www.bbc.com",
                "www.who.int",
                "www.fda.gov",
              ],
              # "blocked_domains": [ # up to 100, useless if you already set allowed_domains
              #  "reddit.com",
              #  "quora.com",
              #  "wikipedia.org",
              #],
          },
          # "user_location": { # to improve search results, you can provide an approximate location of the user
          #     "type": "approximate",
          #     "country": "GB",
          #     "city": "London",
          #     "region": "London",
          # },
          # "external_web_access": False # Default is True, if set to False, the model will not be able to access live content but only cached results
        }
    ],
    reasoning={"effort": "medium"}, # By default summary is None and the reasoning steps content is encrypted, if you want to understand what is happening 
                                       # in the reasoning steps, you can set summary to "concise" or "detailed"
    include=["web_search_call.action.sources"], # returns all the sources used for web search
    input="What was a positive news story from today?",
)

print(f"tokens: {response.usage.total_tokens}")
print(f"output text: {response.output_text}")

32475
Here’s a positive-news item from a trusted domain that’s current as of today (September 3, 2026). I wasn’t able to fetch BBC results due to access restrictions in this tool, but here’s a positive update from WHO’s Newsroom published closest to today:

- September 2, 2026: Joint News Release — Global initiative launched to improve access to mpox vaccines. This marks progress on expanding vaccine access to more people who need it. ([who.int](https://www.who.int/news-room))

- September 1, 2026: Departmental update — WHO moves to expand access to lifesaving sickle cell treatment and care for children. This highlights steps to improve treatment options for a serious condition in kids. ([who.int](https://www.who.int/news-room))

If you’d like, I can keep looking for today’s exact item on these domains as soon as it’s published, or broaden to BBC or FDA sources if you’re okay with other trustworthy outlets. Would you like me to retry for today’s headline sources or stick with WHO for n

Response example (added '...' to short the answer):
```
Response(
    id='resp_0bb76f7cd1fdc95c006a9951d4649487d2bdbb27f1fc2c386c', 
    ... 
    output= [
            ResponseReasoningItem(id='rs_0bb76f7cd1fdc95c006a9951d5195087d2ad27ea6ba2cd76f6', 
                    summary=[], 
                    type='reasoning', 
                    content=[], 
                    encrypted_content='gAAAAA...', 
                    status=None), 
            ResponseFunctionWebSearch(id='ws_0bb76f7cd1fdc95c006a9951d90e0087d28f7976074b3118bc', 
                action=ActionSearch(
                    type='search', 
                    queries=['positive news today BBC', 
                              'BBC good news today', 
                              'site:bbc.com good news today'
                            ], 
                    query='positive news today BBC', 
                    sources=[
                        ActionSearchSource(type='url', url='https://www.who.int/docs/default-source/coronaviru...'), # this search found these urls
                        ActionSearchSource(type='url', url='https://www.who.int/docs/default-source/coronaviru...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/docs/default-source/coronaviru...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/europe/news-room/feature-stori...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/news-room/feature-stories/deta...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/docs/default-source/coronaviru...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/campaigns/world-immunization-w...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/podcasts/episode/science-in-5/...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/westernpacific/newsroom/speech...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/initiatives/health-for-all-fil...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/podcasts/episode/science-in-5/...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/bangladesh/news/detail/11-02-2...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/southeastasia/outbreaks-and-em...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/news-room/feature-stories/deta...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/news/item/24-05-2023-seventy-s...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/nepal/news/detail/23-06-2020-j...'), 
                        ActionSearchSource(type='url', url='https://www.who.int/docs/default-source/eios-gtm-2...'), 
                        ActionSearchSource(type='url', url='https://www.fda.gov/media/124778/download')
                        ]
                ), 
                status='completed', 
                type='web_search_call'), 
            ResponseReasoningItem(...), 
            ResponseFunctionWebSearch(id='ws_0bb76f7cd1fdc95c006a9951db301c87d29c4862c3d45ed25d', 
                        action=ActionSearch(
                            type='search', 
                            queries=['site:who.int today news', 
                                      'site:who.int news 2026'], 
                            query='site:who.int today news', 
                            sources=None                                                                              # this search didn't find anything
                        ), 
                        status='completed', 
                        type='web_search_call'), 
            ResponseReasoningItem(...), 
            ResponseFunctionWebSearch(id='ws_0bb76f7cd1fdc95c006a9951dd444887d2b948ebabe433bd53', ...), 
            ResponseReasoningItem(...), 
            ResponseFunctionWebSearch(id='ws_0bb76f7cd1fdc95c006a9951dff5dc87d28f4e2c357457eb71', ...), 
            ResponseReasoningItem(...), 
            ResponseFunctionWebSearch(id='ws_0bb76f7cd1fdc95c006a9951e15ca887d28194a38b999ed53d', ...), 
            ResponseReasoningItem(...), 
            ResponseFunctionWebSearch(id='ws_0bb76f7cd1fdc95c006a9951e3c2a087d2b19df5b11c643e6c', 
                        action=ActionOpenPage(                                                                       # this is a try to open a page
                          type='open_page', 
                          url='https://www.who.int/news-room'), 
                        status='completed', 
                        type='web_search_call'), 
            ResponseReasoningItem(...), 
            ResponseFunctionWebSearch(...), 
            ResponseReasoningItem(...), 
            ResponseFunctionWebSearch(...), 
            ResponseReasoningItem(...), 
            ResponseFunctionWebSearch(...), 
            ResponseReasoningItem(...), 
            ResponseOutputMessage(id='msg_0bb76f7cd1fdc95c006a9951ff7e4c87d2b5cc3fb2e565ad8d', 
                        content=[
                            ResponseOutputText(
                                annotations=[
                                    AnnotationURLCitation(end_index=475, start_index=433, title='Newsroom', type='url_citation', url='https://www.who.int/news-room'), 
                                    AnnotationURLCitation(end_index=732, start_index=690, title='Newsroom', type='url_citation', url='https://www.who.int/news-room')
                                ], 
                                text='Here’s a positive-news item from a trusted domain ... with WHO for now?', 
                                type='output_text', 
                                logprobs=[])
                            ], 
                            role='assistant', 
                            status='completed', 
                            type='message', 
                            phase=None)
            ], 
    output_text: 'Here’s a positive-news item from a trusted domain ... with WHO for now?',
    ...
    tool_usage={'image_gen': {...}, 'web_search': {'num_requests': 5}})
    usage=ResponseUsage(input_tokens=26993, ..., output_tokens=5482, output_tokens_details=OutputTokensDetails(reasoning_tokens=4992), total_tokens=32475, ...),
```

### GPT-5.6-terra
gpt-5.6-terra took 5k tokens to produce the response

In [8]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

response = client.responses.create(
    model="gpt-5.6-terra",
    tools=[
        {
          "type": "web_search",
          "search_context_size": "low",
          "filters": {
              "allowed_domains": [
                "www.bbc.com",
                "www.who.int",
                "www.fda.gov",
              ],
          }
        }
    ],
    reasoning={"effort": "medium", "summary":"concise"},
    include=["web_search_call.action.sources"],
    input="What was a positive news story from today?"
)

print(f"tokens: {response.usage.total_tokens}")
print(f"output text: {response.output_text}")

tokens: 4953
output text: I can’t reliably identify a positive story from **today, September 3, 2026** under the required source limits: the permitted BBC/WHO/FDA search results were unavailable or did not return a verifiable story.


Response example (the reasoning is way shorter and model took just one websearch to decide it can't access info):
```
output=[
    ResponseReasoningItem(id='...', 
                          summary=[Summary(text='**Searching trusted news sources**', type='summary_text')], 
                          type='reasoning', 
                          content=[], 
                          encrypted_content='gAAAAABqm...', 
                          status=None), 
    ResponseFunctionWebSearch(id='...', action=ActionSearch(
                          type='search', 
                          queries=['site:bbc.com/news "3 September 2026" positive news', 
                                   'site:who.int "3 September 2026" news', 
                                   'site:fda.gov "September 3, 2026" news'], 
                          query='site:bbc.com/news "3 September 2026" positive news', 
                          sources=None), status='completed', type='web_search_call'), 
    ResponseReasoningItem(id='...', 
                          summary=[Summary(text='**Handling inaccessible search results**', type='summary_text')], 
                          type='reasoning', 
                          content=[], 
                          encrypted_content='...', 
                          status=None), 
    ResponseOutputMessage(id='...', 
                          content=[
                            ResponseOutputText(
                                annotations=[], 
                                text='I can’t reliably identify ...', 
                                type='output_text', 
                                logprobs=[])
                            ], 
                          role='assistant', 
                          status='completed', 
                          type='message', 
                          phase='final_answer')
],
usage=ResponseUsage(input_tokens=4663, 
                    input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=4383), 
                    output_tokens=290, 
                    output_tokens_details=OutputTokensDetails(reasoning_tokens=238), 
                    total_tokens=4953, 
                    compute_units=None), 
tool_usage={'image_gen': {...}, 
            'web_search': {'num_requests': 1}})
```  

### GPT-5.6-terra without domain filters
gpt-5.6-terra this time return a news using 19k tokens.

This time it search on the web and opens one of the pages found.

In [10]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

response = client.responses.create(
    model="gpt-5.6-terra",
    tools=[
        {
          "type": "web_search",
          "search_context_size": "low",
        }
    ],
    reasoning={"effort": "medium", "summary":"concise"},
    include=["web_search_call.action.sources"],
    input="What was a positive news story from today?"
)

print(f"tokens: {response.usage.total_tokens}")
print(f"output text: {response.output_text}")

tokens: 19598
output text: One uplifting story from **Thursday, September 3, 2026**: in Prospect Park, Pennsylvania, neighbors, local workers, and a FedEx driver rushed to help after an e-bike rider was pinned beneath a Volkswagen Beetle. Together, they lifted the car and pulled her free; police said their quick action helped save her life. She was hospitalized in stable condition. ([goodmorningamerica.com](https://www.goodmorningamerica.com/news/story/good-samaritans-helped-lift-car-free-woman-trapped-136169340))


```
...
output=[
    ResponseReasoningItem(..., summary=[Summary(text='**Determining current date and news retrieval**', type='summary_text')], ...), 
    ResponseFunctionWebSearch(id='...', 
        action=ActionSearch(
            type='search', 
            queries=['positive news September 3 2026 good news', 
                     'site:apnews.com September 3 2026 positive news', 
                     'site:reuters.com September 3 2026 uplifting good news'], 
            query='positive news September 3 2026 good news', 
            sources=[
                ActionSearchSource(type='url', url='https://apnews.com/article/27b78c349725ac744c96a6b8b8167bae'), 
                ActionSearchSource(type='url', url='https://www.theguardian.com/news/2026/sep/03/all'), 
                ActionSearchSource(type='url', url='https://www.goodmorningamerica.com/'), 
                ActionSearchSource(type='url', url='https://www.theguardian.com/world/2026/sep/03/all'), 
                ActionSearchSource(type='url', url='https://www.csmonitor.com/Daily/2026/20260903'), 
                ActionSearchSource(type='url', url='https://www.democracynow.org/2026/9/3/headlines'), 
                ActionSearchSource(type='url', url='https://www.manxradio.com/news/good-news/?d=2026-09-03'), 
                ActionSearchSource(type='url', url='https://good-wire.com/'), 
                ActionSearchSource(type='url', url='https://goodvibetoday.com/'), 
                ActionSearchSource(type='url', url='https://onlyhappynews.com/'), 
                ActionSearchSource(type='url', url='https://happyreaddaily.com/'), 
                ActionSearchSource(type='url', url='https://johnplusnews.com/'), 
                ActionSearchSource(type='url', url='https://www.goodnewsnetwork.org/'), 
                ActionSearchSource(type='url', url='https://apnews.com/article/18ffac50eafd63c550f1b775af09b902'), 
                ActionSearchSource(type='url', url='https://www.reddit.com/r/globalcheckin/comments/1w5xt44/how_is_your_country_doing_today_september_03_2026/'), 
                ActionSearchSource(type='url', url='https://en.wikipedia.org/wiki/September_1926')]), 
            status='completed', 
            type='web_search_call'), 
    ResponseReasoningItem(..., summary=[Summary(text='**Considering Good Samaritan example**', type='summary_text')], ...), 
    ResponseFunctionWebSearch(..., 
            action=ActionOpenPage(
                type='open_page', 
                url='https://www.goodmorningamerica.com/'), 
                status='completed', 
                type='web_search_call'), 
    ResponseReasoningItem(..., summary=[Summary(text='**Accessing article**', type='summary_text')], ...), 
    ResponseFunctionWebSearch(..., 
            action=ActionOpenPage(
                type='open_page', 
                url='https://www.goodmorningamerica.com/'), 
                status='completed', 
                type='web_search_call'), 
    ResponseOutputMessage(..., 
            content=[
                ResponseOutputText(annotations=[AnnotationURLCitation(end_index=492, 
                                                                      start_index=358, 
                                                                      title='Good Samaritans lift car to free woman trapped underneath after e-bike crash in Pennsylvania - Good Morning America', 
                                                                      type='url_citation', 
                                                                      url='https://www.goodmorningamerica.com/news/story/good-samaritans-helped-lift-car-free-woman-trapped-136169340')], 
                                                                      text='One uplifting story from **Thursday, September 3, 2026**: in Prospect Park, Pennsylvania, neighbors, local workers, and a FedEx driver rushed to help after an e-bike rider was pinned beneath a Volkswagen Beetle. Together, they lifted the car and pulled her free; police said their quick action helped save her life. She was hospitalized in stable condition. ([goodmorningamerica.com](https://www.goodmorningamerica.com/news/story/good-samaritans-helped-lift-car-free-woman-trapped-136169340))', 
                                                                      type='output_text', 
                                                                      logprobs=[])], 
                role='assistant', status='completed', type='message', phase='final_answer')
            ], 
    completed_at=1788448579.0, 
    usage=ResponseUsage(input_tokens=19264, 
                        input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=4383), 
                        output_tokens=334, 
                        output_tokens_details=OutputTokensDetails(reasoning_tokens=240), 
                        total_tokens=19598, 
                        compute_units=None), 
    tool_usage={'image_gen': {...}, 
                'web_search': {'num_requests': 1}}})
```